In [ ]:
import ee
import geemap
ee.Authenticate()
ee.Initialize()

# Map Initialization
Map = geemap.Map()

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
panama_fc = countries.filter(ee.Filter.eq("ADM0_NAME", "Panama"))
panama_geom = panama_fc.geometry()

Map.centerObject(panama_geom, 7)

terraclim = (
    ee.ImageCollection("IDAHO_EPSCOR/TERRACLIMATE")
    .filterDate("2014-01-01", "2024-12-01")
    .filterBounds(panama_geom)
    .select(["tmmx", "tmmn"])
)

# Function to calculate yearly metrics with scaling applied after filtering
def calculate_yearly_metrics(year):

    def scale_and_aggregate(var, reducer):
        data = terraclim.select(var).filter(ee.Filter.calendarRange(year, year, "year"))
        if var in ["tmmn", "tmmx"]:
            data = data.map(lambda img: img.multiply(0.1))
        if var in ["tmmn", "tmmx"]:
            data = data.reduce(reducer).float().rename(f"{var}_{year}")
        else:
            data = data.reduce(reducer).toInt16().rename(f"{var}_{year}")
        return data

    # Aggregate mean temperature (average of max and min)
    maxtemp = scale_and_aggregate("tmmx", ee.Reducer.mean())
    mintemp = scale_and_aggregate("tmmn", ee.Reducer.mean())
    yearly_temperature = maxtemp.addBands(mintemp).reduce(ee.Reducer.mean()).float().rename(f"temp_{year}")

    return ee.Image.cat([yearly_temperature])

# Create a dictionary for variables with filtering and scaling applied after
vars = {var: terraclim.select(var) for var in ["tmmx", "tmmn"]}

# Calculate yearly metrics and combine into a single image
yearly_terraclim = ee.Image.cat([calculate_yearly_metrics(year) for year in range(2014, 2025)])

# Export the final image
task = ee.batch.Export.image.toDrive(
    image=yearly_terraclim,
    description='Panama_Temperature_Javier',
    folder='GEE_Exports',
    fileNamePrefix='panama_temperature',
    region=panama_geom,
    scale=4638.3,  # Terraclim resolution is 4638.3 m
    maxPixels=1e13
)

task.start()


In [ ]:
map = geemap.Map()
map.addLayer(yearly_terraclim)
map


In [ ]:
dataset = ee.ImageCollection('IDAHO_EPSCOR/TERRACLIMATE').filter(
    ee.Filter.date('2017-07-01', '2017-08-01')
)
maximum_temperature = dataset.select('tmmx')

maximum_temperature_vis = {
    'min': -300.0,
    'max': 300.0,
    'palette': [
        '1a3678',
        '2955bc',
        '5699ff',
        '8dbae9',
        'acd1ff',
        'caebff',
        'e5f9ff',
        'fdffb4',
        'ffe6a2',
        'ffc969',
        'ffa12d',
        'ff7c1f',
        'ca531a',
        'ff0000',
        'ab0000',
    ],
}

m = geemap.Map()
m.set_center(71.72, 52.48, 3)
m.add_layer(
    maximum_temperature, maximum_temperature_vis, 'Maximum Temperature'
)
m

Map(center=[52.48, 71.72], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…